# Lezione 12 — Overfitting: quando il modello impara a memoria

**Come si usa questo notebook.** Come sempre. Prerequisiti: Lezioni 10-11.
Chiude il primo blocco della Fase 2 — e il progetto apre, una sola volta,
il suo test set.

**Cosa saprai fare alla fine:** riconoscere l'overfitting dalle curve di
apprendimento, contenerlo con early stopping e dropout, e condurre una
valutazione finale onesta.

## Parte 1 — Teoria: memorizzare è facile, generalizzare no

Lo studente della Lezione 3 che imparava a memoria le soluzioni? Una rete
con abbastanza capacità fa esattamente questo: con più parametri che
esempi, può ottenere loss ~0 sul train **memorizzando** — e fallire sui
dati nuovi. È l'**overfitting**, e non è un guasto: è il comportamento
naturale di un ottimizzatore potente su pochi dati. Va *gestito*, non
sperato via.

Lo strumento diagnostico sono le **curve di apprendimento**: loss di train
e di validation lungo le epoche.

- entrambe scendono → il modello sta ancora imparando cose vere;
- il train continua a scendere ma la **validation risale** → da quel punto
  in poi il modello sta memorizzando rumore. Il punto di svolta è il
  momento giusto per fermarsi.

Le contromisure standard:

1. **Early stopping** — fermati quando la validation smette di migliorare
   (con una `patience`: qualche epoca di tolleranza al rumore). Semplice e
   quasi sempre la prima cosa giusta da fare.
2. **Dropout** — durante il training, a ogni passo spegni a caso una
   frazione dei neuroni (es. 30%). Ogni neurone impara a non dipendere
   troppo dagli altri; in inferenza sono tutti accesi. Una forma di
   robustezza forzata.
3. **Più dati / meno capacità / regolarizzazione dei pesi** — le altre
   leve, che incontrerai strada facendo.

Prima la diagnosi, con un caso costruito apposta: rete grande, pochi dati.

In [ ]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'   # silenzia i log informativi di TF

import numpy as np
import pandas as pd
import keras

keras.utils.set_random_seed(42)            # riproducibilita' (Lezione 3!)
print('Keras', keras.__version__)

In [ ]:
from pathlib import Path

processed = Path('..') / 'datasets' / 'processed'
insiemi = {n: pd.read_csv(processed / f'memory_features_{n}.csv')
           for n in ['train', 'val', 'test']}
X = {n: f.drop(columns='target').to_numpy().astype('float32') for n, f in insiemi.items()}
y = {n: f['target'].to_numpy() for n, f in insiemi.items()}

keras.utils.set_random_seed(42)
esagerata = keras.Sequential([
    keras.layers.Dense(128, activation='relu'),
    keras.layers.Dense(128, activation='relu'),
    keras.layers.Dense(4, activation='softmax'),
])
esagerata.compile(loss='sparse_categorical_crossentropy', optimizer='adam',
                  metrics=['accuracy'])
storia = esagerata.fit(X['train'], y['train'], epochs=300,
                       validation_data=(X['val'], y['val']), verbose=0)
print('parametri della rete:', esagerata.count_params(), '   esempi di train:', len(y['train']))

In [ ]:
import matplotlib.pyplot as plt

fig, asse = plt.subplots(figsize=(8, 3.5))
asse.plot(storia.history['loss'], label='loss train')
asse.plot(storia.history['val_loss'], label='loss validation')
asse.set_xlabel('epoca')
asse.legend()
asse.set_title('Overfitting in diretta: il train scende, la validation risale')
plt.tight_layout()
plt.show()

Eccolo: 18mila parametri contro 213 esempi. La curva di train scivola verso
zero (memorizzazione riuscita), quella di validation tocca il minimo e poi
**risale** — ogni epoca oltre quel punto peggiora il modello sui dati veri.
Questo grafico è il più importante del machine learning pratico: impara a
chiederlo a ogni training.

## Parte 2 — Esercizio guidato

Il tuo compito: riaddestra la **stessa** rete esagerata, ma con
l'early stopping che ferma il training quando `val_loss` non migliora da
20 epoche e ripristina i pesi migliori:

```python
stop = keras.callbacks.EarlyStopping(monitor='val_loss', patience=20,
                                     restore_best_weights=True)
# ...fit(..., callbacks=[stop])
```

Poi confronta: a che epoca si è fermato? L'accuratezza di validation è
migliore della versione da 300 epoche?

**Prova tu:**

In [ ]:
keras.utils.set_random_seed(42)

# Scrivi qui: stessa architettura 128-128-4, compile identico,
# fit con callbacks=[stop] e epochs=300

pass

### Soluzione spiegata

- il callback osserva `val_loss` a fine epoca: se non migliora per
  `patience` epoche consecutive, ferma il training;
- `restore_best_weights=True` è essenziale: senza, ti tieni i pesi
  dell'ultima epoca (già in overfitting), non quelli del punto migliore;
- nota il principio: la decisione di fermarsi usa la **validation** — mai
  il test (Lezione 3: ogni scelta consuma l'insieme su cui la fai).

In [ ]:
keras.utils.set_random_seed(42)
controllata = keras.Sequential([
    keras.layers.Dense(128, activation='relu'),
    keras.layers.Dense(128, activation='relu'),
    keras.layers.Dense(4, activation='softmax'),
])
controllata.compile(loss='sparse_categorical_crossentropy', optimizer='adam',
                    metrics=['accuracy'])
stop = keras.callbacks.EarlyStopping(monitor='val_loss', patience=20,
                                     restore_best_weights=True)
storia_es = controllata.fit(X['train'], y['train'], epochs=300,
                            validation_data=(X['val'], y['val']),
                            callbacks=[stop], verbose=0)

print(f'fermata dopo {len(storia_es.history["loss"])} epoche (su 300)')
print(f"validation senza controllo : {esagerata.evaluate(X['val'], y['val'], verbose=0)[1]:.0%}")
print(f"validation con early stop  : {controllata.evaluate(X['val'], y['val'], verbose=0)[1]:.0%}")

## Parte 3 — Il progetto: Memory AI Lab, passo 12 — la resa dei conti

Chiudiamo il blocco come si chiude un esperimento vero:

1. il modello **finale** usa ciò che hai imparato: capacità moderata,
   dropout, early stopping;
2. si sceglie tutto su train+validation;
3. e poi — per la prima e **unica** volta — si apre il test set (Lezione
   3: il test si usa una volta, alla fine). Il numero che esce è la stima
   onesta di come il classificatore si comporterebbe su memorie mai viste.

In [ ]:
keras.utils.set_random_seed(42)
finale = keras.Sequential([
    keras.layers.Dense(32, activation='relu'),
    keras.layers.Dropout(0.3),
    keras.layers.Dense(4, activation='softmax'),
])
finale.compile(loss='sparse_categorical_crossentropy', optimizer='adam',
               metrics=['accuracy'])
finale.fit(X['train'], y['train'], epochs=300,
           validation_data=(X['val'], y['val']),
           callbacks=[keras.callbacks.EarlyStopping(monitor='val_loss', patience=20,
                                                    restore_best_weights=True)],
           verbose=0)

W0, b0 = np.load(processed / 'softmax_baseline_W.npy'), np.load(processed / 'softmax_baseline_b.npy')
per_insieme = {}
for n in ['train', 'val', 'test']:
    keras_acc = finale.evaluate(X[n], y[n], verbose=0)[1]
    base_acc = ((X[n] @ W0 + b0).argmax(axis=1) == y[n]).mean()
    per_insieme[n] = (base_acc, keras_acc)

print(f'{"insieme":8} {"baseline Fase 0":>16} {"rete finale":>12}')
for n, (base, ker) in per_insieme.items():
    print(f'{n:8} {base:16.0%} {ker:12.0%}')

Path('../models').mkdir(exist_ok=True)
finale.save('../models/memory_type_classifier.keras')
print('\nModello salvato in models/memory_type_classifier.keras')

Leggi la tabella dal basso: la riga **test** è il verdetto — mai visto
prima d'ora né dal modello né da te. E qui il corso ti deve un'ultima
lezione di onestà: il nostro test ha **15 esempi**. Con 15 esempi, ogni
memoria vale quasi 7 punti percentuali: una differenza di 1-2 memorie tra
rete e baseline è **rumore**, non un verdetto. Per questo su test piccoli
si riportano gli intervalli di incertezza, e le conclusioni forti si
rimandano a quando ci sono più dati — il Memory AI Lab, crescendo,
riempirà anche il test.

La conclusione onesta resta quella della Lezione 10: con 7 feature povere
il modello non è il collo di bottiglia. Il classificatore comunque c'è,
funziona, ed è salvato: è il primo **modello di produzione** del Memory AI
Lab. La Fase 3 gli darà occhi veri sul testo (embedding) — e lì si riparte
dal protocollo che ormai possiedi: baseline, validation, test sigillato.

## Cosa hai imparato — e il primo blocco Fase 2 è completo

- L'overfitting è il comportamento naturale di un modello potente su pochi
  dati: si **diagnostica** con le curve train/validation.
- **Early stopping**: fermarsi al minimo della validation, coi pesi
  migliori ripristinati. **Dropout**: robustezza spegnendo neuroni a caso.
- Ogni decisione (quando fermarsi, quale architettura) si prende sulla
  validation; il **test si apre una volta sola**, alla fine.
- Su un test piccolo la stima è **rumorosa**: differenze di pochi esempi
  non sono verdetti. Dichiara l'incertezza insieme al numero.
- Il progetto ha il suo primo modello salvato e una stima onesta delle sue
  capacità.

## Quiz

1. La loss di train è 0.05, quella di validation 1.4 e cresce. Cosa sta
   succedendo e quali due contromisure provi per prime?
2. Perché `restore_best_weights=True` è importante?
3. Dopo la valutazione sul test, ti viene voglia di provare "solo un'altra
   architettura" e rivalutare sul test. Perché no, e cosa faresti invece?

<details>
<summary><b>Apri le risposte</b></summary>

1. Overfitting conclamato: il modello memorizza il train. Prime mosse:
   early stopping sulla val_loss e riduzione di capacità (o dropout).
2. Senza, il modello finale ha i pesi dell'ultima epoca — che per
   definizione è oltre il punto migliore, visto che l'early stopping
   scatta dopo `patience` epoche di peggioramento.
3. Il secondo sguardo al test lo trasforma in un validation: la stima
   smette di essere onesta (Lezione 3: ottimizzare sul test lo consuma).
   Si torna a scegliere su validation; il test si riapre solo per il
   verdetto del nuovo esperimento, e la disciplina va dichiarata prima.
</details>

## Fonti

- TensorFlow, *Overfit and underfit*:
  https://www.tensorflow.org/tutorials/keras/overfit_and_underfit
- Keras, `EarlyStopping`:
  https://keras.io/api/callbacks/early_stopping/
- Keras, `Dropout`: https://keras.io/api/layers/regularization_layers/dropout/

## Prossimo passo del corso

Il classificatore vede solo lunghezze e flag: è cieco al **significato**
del testo. La Fase 3 — tokenizzazione, embedding, similarità — gli dà gli
occhi. È il cuore del Memory AI Lab: trasformare frasi in vettori
confrontabili, e cercare memorie per significato.